In [1]:
!pip install onnx
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 135.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.8 MB/s eta 0:00:00


In [2]:
import os
import time
from contextlib import nullcontext

import torch
import torch.nn as nn
import onnxruntime as ort
from transformers import AutoModel, AutoTokenizer

In [3]:
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.8.0+cu126
CUDA version: 12.6
GPU: Tesla T4


Load model and tokenizer

In [4]:
# Load the sentence transformer model
model_name = "sentence-transformers/multi-qa-mpnet-base-cos-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Create Sample Input

In [5]:
# Sample text for inference
sample_text = "But I must explain to you how all this mistaken idea of denouncing pleasure and praising pain was born and I will give you a complete account of the system, and expound the actual teachings of the great explorer of the truth, the master-builder of human happiness."

# Tokenize the input
inputs = tokenizer(sample_text, padding=True, truncation=True, return_tensors="pt")

### Exercise 1: evaluation mode

In [6]:
def measure_inference(model, inputs, num_runs=100, warmup=10, mode='none', sync_cuda=False):
    if mode == 'no_grad':
        ctx = torch.no_grad()
    elif mode == 'inference_mode':
        ctx = torch.inference_mode()
    else:
        ctx = nullcontext()

    # Warmup runs
    for _ in range(warmup):
        with ctx:
            _ = model(**inputs)

    if sync_cuda and torch.cuda.is_available():
        torch.cuda.synchronize()

    # Measure time
    start_time = time.time()
    for _ in range(num_runs):
        with ctx:
            _ = model(**inputs)
    if sync_cuda and torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()

    return (end_time - start_time) / num_runs * 1000  # Convert to ms

1. Testing: no optimizations

In [7]:
model_no_opt = AutoModel.from_pretrained(model_name)
time_no_opt = measure_inference(model_no_opt, inputs, mode='none')

2. Testing: model.eval()

In [8]:
model_eval = AutoModel.from_pretrained(model_name)
model_eval.eval()
time_eval = measure_inference(model_eval, inputs, mode='none')

3. Testing: model.eval() + torch.no_grad()

In [9]:
model_no_grad = AutoModel.from_pretrained(model_name)
model_no_grad.eval()
time_no_grad = measure_inference(model_no_grad, inputs, mode='no_grad')

4. Testing: model.eval() + torch.inference_mode()

In [10]:
model_inference = AutoModel.from_pretrained(model_name)
model_inference.eval()
time_inference = measure_inference(model_inference, inputs, mode='inference_mode')

Results from ex. 1

In [11]:
print(f"1. No optimizations:           {time_no_opt:.3f} ms (baseline)")
print(f"2. model.eval():               {time_eval:.3f} ms ({time_no_opt/time_eval:.3f}x speedup)")
print(f"3. eval() + no_grad():         {time_no_grad:.3f} ms ({time_no_opt/time_no_grad:.3f}x speedup)")
print(f"4. eval() + inference_mode():  {time_inference:.3f} ms ({time_no_opt/time_inference:.3f}x speedup)")


1. No optimizations:           273.690 ms (baseline)
2. model.eval():               175.517 ms (1.559x speedup)
3. eval() + no_grad():         168.411 ms (1.625x speedup)
4. eval() + inference_mode():  169.815 ms (1.612x speedup)


### Exercise 2: PyTorch model compilation

In [12]:
model_compiled = AutoModel.from_pretrained(model_name)
model_compiled.eval()

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [13]:
# Measure compilation + warmup time
start_compile = time.time()
compiled_model = torch.compile(model_compiled)

# Warmup run (triggers actual compilation)
with torch.inference_mode():
    _ = compiled_model(**inputs)

compile_time = time.time() - start_compile
time_compiled = measure_inference(compiled_model, inputs, mode='inference_mode')

Results from ex. 2

In [14]:
print(f"Compilation + warmup time:  {compile_time:.3f} seconds")
print(f"Baseline (no opt):          {time_no_opt:.3f} ms")
print(f"Eval + inference_mode:      {time_inference:.3f} ms")
print(f"Compiled model:             {time_compiled:.3f} ms")
print(f"Speedup vs baseline:        {time_no_opt / time_compiled:.3f}x")
print(f"Speedup vs inference_mode:  {time_inference / time_compiled:.3f}x")

Compilation + warmup time:  62.630 seconds
Baseline (no opt):          273.690 ms
Eval + inference_mode:      169.815 ms
Compiled model:             173.656 ms
Speedup vs baseline:        1.576x
Speedup vs inference_mode:  0.978x


### Exercise 3: quantization

In [15]:
# Ensure model is on CPU
model_cpu = AutoModel.from_pretrained(model_name)
model_cpu.eval()
model_cpu = model_cpu.cpu()

Quantize model

In [16]:
model_quantized = torch.ao.quantization.quantize_dynamic(
    model_cpu,
    {nn.Linear},
    dtype=torch.qint8
)

/tmp/ipython-input-2428617119.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_quantized = torch.ao.quantization.quantize_dynamic(


Save both models and compare sizes

In [17]:
# Save original model
torch.save(model_cpu.state_dict(), "model_original.pth")
original_size = os.path.getsize("model_original.pth")

# Save quantized model
torch.save(model_quantized.state_dict(), "model_quantized.pth")
quantized_size = os.path.getsize("model_quantized.pth")

In [18]:
print(f"Original model size:  {original_size / 1024 / 1024:.2f} MB")
print(f"Quantized model size: {quantized_size / 1024 / 1024:.2f} MB")
print(f"Size reduction:       {original_size / quantized_size:.2f}x")

Original model size:  417.73 MB
Quantized model size: 173.10 MB
Size reduction:       2.41x


Compare inference speeds on CPU

In [19]:
time_cpu_original = measure_inference(model_cpu, inputs, mode='inference_mode')
time_cpu_quantized = measure_inference(model_quantized, inputs, mode='inference_mode')

Results from ex. 3

In [20]:
print(f"Original model size:              {original_size / 1024 / 1024:.2f} MB")
print(f"Quantized model size:             {quantized_size / 1024 / 1024:.2f} MB")
print(f"Size reduction:                   {original_size / quantized_size:.2f}x")
print(f"Original model inference speed:   {time_cpu_original:.3f} ms")
print(f"Quantized model inference speed:  {time_cpu_quantized:.3f} ms")
print(f"Speedup:                          {time_cpu_original / time_cpu_quantized:.3f}x")

Original model size:              417.73 MB
Quantized model size:             173.10 MB
Size reduction:                   2.41x
Original model inference speed:   179.903 ms
Quantized model inference speed:  84.594 ms
Speedup:                          2.127x


### Exercise 4: GPU optimization with torch.compile modes

In [21]:
device = torch.device('cuda')

# Move model and inputs to GPU
model_gpu = AutoModel.from_pretrained(model_name).to(device)
model_gpu.eval()
inputs_gpu = {k: v.to(device) for k, v in inputs.items()}

1. Testing: torch.compile() with default settings

In [22]:
model_compile_default = AutoModel.from_pretrained(model_name).to(device)
model_compile_default.eval()
model_compile_default = torch.compile(model_compile_default)

# Warmup compilation
with torch.inference_mode():
    _ = model_compile_default(**inputs_gpu)

time_default = measure_inference(model_compile_default, inputs_gpu, mode='inference_mode', sync_cuda=True)

W1118 20:06:52.840000 1311 torch/_inductor/utils.py:1436] [0/1] Not enough SMs to use max_autotune_gemm mode


2. Testing: torch.compile() with mode='max-autotune'

In [23]:
model_max_autotune = AutoModel.from_pretrained(model_name).to(device)
model_max_autotune.eval()
model_max_autotune = torch.compile(model_max_autotune, mode="max-autotune")

with torch.inference_mode():
    _ = model_max_autotune(**inputs_gpu)

time_max_autotune = measure_inference(model_max_autotune, inputs_gpu, mode='inference_mode', sync_cuda=True)

AUTOTUNE addmm(57x768, 57x768, 768x768)
strides: [0, 1], [768, 1], [1, 768]
dtypes: torch.float32, torch.float32, torch.float32
  addmm 0.0737 ms 100.0% 
  bias_addmm 0.0991 ms 74.4% 
SingleProcess AUTOTUNE benchmarking takes 0.0345 seconds and 0.0003 seconds precompiling for 2 choices


3. Testing: torch.compile() with mode='max-autotune-no-cudagraph'

In [24]:
model_no_cudagraphs = AutoModel.from_pretrained(model_name).to(device)
model_no_cudagraphs.eval()
model_no_cudagraphs = torch.compile(model_no_cudagraphs, mode="max-autotune-no-cudagraphs")

# Warmup compilation
with torch.inference_mode():
    _ = model_no_cudagraphs(**inputs_gpu)

time_no_cudagraphs = measure_inference(model_no_cudagraphs, inputs_gpu, mode='inference_mode', sync_cuda=True)

4. Testing with different input sizes

In [25]:
test_texts = [
    "Short text.",
    "This is a medium length text for testing the model inference with different input sizes.",
    "This is a much longer text designed to test how the model handles longer sequences. " * 5
]

for i, text in enumerate(test_texts, 1):
    print(f"Test {i}: {len(text)} characters")
    test_inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    test_inputs_gpu = {k: v.to(device) for k, v in test_inputs.items()}

    # Test max-autotune
    time_test = measure_inference(model_max_autotune, test_inputs_gpu, num_runs=50, mode='inference_mode', sync_cuda=True)
    print(f"   max-autotune: {time_test:.3f} ms")

    # Test max-autotune-no-cudagraphs
    time_test = measure_inference(model_no_cudagraphs, test_inputs_gpu, num_runs=50, mode='inference_mode', sync_cuda=True)
    print(f"   max-autotune-no-cudagraphs: {time_test:.3f} ms")

Test 1: 11 characters


/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7095: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
AUTOTUNE addmm(5x768, 5x768, 768x768)
strides: [0, 1], [768, 1], [1, 768]
dtypes: torch.float32, torch.float32, torch.float32
  addmm 0.0615 ms 100.0% 
  bias_addmm 0.0840 ms 73.2% 
SingleProcess AUTOTUNE benchmarking takes 0.0519 seconds and 0.0003 seconds precompiling for 2 choices


   max-autotune: 4.153 ms


/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7095: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(


   max-autotune-no-cudagraphs: 3.921 ms
Test 2: 88 characters
   max-autotune: 3.984 ms
   max-autotune-no-cudagraphs: 3.998 ms
Test 3: 420 characters
   max-autotune: 8.785 ms
   max-autotune-no-cudagraphs: 9.005 ms


Results of ex. 4

In [26]:
print(f"Default compile:            {time_default:.3f} ms (baseline)")
print(f"max-autotune:               {time_max_autotune:.3f} ms ({time_default/time_max_autotune:.3f}x speedup)")
print(f"max-autotune-no-cudagraphs: {time_no_cudagraphs:.3f} ms ({time_default/time_no_cudagraphs:.3f}x speedup)")

Default compile:            5.787 ms (baseline)
max-autotune:               6.248 ms (0.926x speedup)
max-autotune-no-cudagraphs: 6.443 ms (0.898x speedup)


### Exercise 5: changing numerical precision

1. Testing: Full precision (float32)

In [27]:
model_fp32 = AutoModel.from_pretrained(model_name).to(device)
model_fp32.eval()
inputs_fp32 = {k: v.to(device) for k, v in inputs.items()}

time_fp32 = measure_inference(model_fp32, inputs_fp32, mode='inference_mode', sync_cuda=True)

2. Testing: Half precision (float16)

In [28]:
model_fp16 = AutoModel.from_pretrained(model_name).to(device).half()
model_fp16.eval()

time_fp16 = measure_inference(model_fp16, inputs_fp32, mode='inference_mode', sync_cuda=True)

3. Testing: Automatic mixed precision (torch.autocast)

In [29]:
model_amp = AutoModel.from_pretrained(model_name).to(device)
model_amp.eval()

time_amp = measure_inference(model_amp, inputs_fp32, mode='inference_mode', sync_cuda=True)

In [30]:
print(f"Full precision (float32):        {time_fp32:.3f} ms (baseline)")
print(f"Half precision (float16):        {time_fp16:.3f} ms ({time_fp32/time_fp16:.3f}x speedup)")
print(f"Automatic mixed precision (AMP): {time_amp:.3f} ms ({time_fp32/time_amp:.3f}x speedup)")

Full precision (float32):        7.994 ms (baseline)
Half precision (float16):        8.076 ms (0.990x speedup)
Automatic mixed precision (AMP): 7.964 ms (1.004x speedup)


### Exercise 6: ONNX export and optimization

In [31]:
# Prepare model for export (CPU, eval mode)
model_export = AutoModel.from_pretrained(model_name)
model_export.eval()
model_export = model_export.cpu()

In [32]:
# Prepare sample input for tracing
sample_input = tokenizer(
    "This is a sample input text for ONNX export.",
    padding=True,
    truncation=True,
    return_tensors="pt",
)

In [33]:
# Export to ONNX
onnx_path = "model.onnx"
torch.onnx.export(
    model_export,
    (sample_input["input_ids"], sample_input["attention_mask"]),
    onnx_path,
    opset_version=17,
    input_names=["input_ids", "attention_mask"],
    output_names=["output"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "output": {0: "batch_size"},
    },
)

/tmp/ipython-input-4156519402.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


Test 1: Online optimization (cold start)

In [34]:
# Measure cold start time (session creation + first inference)
start_cold = time.time()
ort_session_online = ort.InferenceSession(
    onnx_path,
    providers=["CPUExecutionProvider"]
)

# Prepare input for ONNX
sample_onnx = tokenizer(
    sample_text,
    padding=True,
    truncation=True,
    return_tensors="np"
)

inputs_onnx = {
    "input_ids": sample_onnx["input_ids"],
    "attention_mask": sample_onnx["attention_mask"],
}

# First inference (part of cold start)
_ = ort_session_online.run(None, inputs_onnx)
cold_start_time = time.time() - start_cold

In [35]:
# Measure inference time for online optimization
def measure_onnx_inference(session, inputs, num_runs=100, warmup=10):
    # Warmup
    for _ in range(warmup):
        _ = session.run(None, inputs)

    # Measure
    start = time.time()
    for _ in range(num_runs):
        _ = session.run(None, inputs)
    end = time.time()

    return (end - start) / num_runs * 1000

In [36]:
time_onnx_online = measure_onnx_inference(ort_session_online, inputs_onnx)

Test 2: Offline optimization

In [37]:
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
sess_options.optimized_model_filepath = "model_optimized.onnx"

start_offline_opt = time.time()
_ = ort.InferenceSession(onnx_path, sess_options)
offline_opt_time = time.time() - start_offline_opt

In [38]:
# Load the optimized model (cold start for offline optimized)
start_cold_offline = time.time()

sess_options_load = ort.SessionOptions()
sess_options_load.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL

ort_session_offline = ort.InferenceSession(
    "model_optimized.onnx",
    sess_options=sess_options_load,
    providers=["CPUExecutionProvider"]
)

# First inference
_ = ort_session_offline.run(None, inputs_onnx)
cold_start_offline_time = time.time() - start_cold_offline

In [39]:
# Measure inference time for offline optimization
time_onnx_offline = measure_onnx_inference(ort_session_offline, inputs_onnx)

Results of ex. 6

In [40]:
print(f"Cold Start Times:")
print(f"Online optimization:            {cold_start_time:.3f} seconds")
print(f"Offline optimization (loading): {cold_start_offline_time:.3f} seconds")
print(f"Offline opt speedup:            {cold_start_time / cold_start_offline_time:.3f}x")

Cold Start Times:
Online optimization:            0.918 seconds
Offline optimization (loading): 0.913 seconds
Offline opt speedup:            1.005x


In [41]:
print(f"Inference Times (CPU):")
print(f"PyTorch (best from Ex1): {time_inference:.3f} ms")
print(f"ONNX online:             {time_onnx_online:.3f} ms")
print(f"ONNX offline:            {time_onnx_offline:.3f} ms")

Inference Times (CPU):
PyTorch (best from Ex1): 169.815 ms
ONNX online:             117.416 ms
ONNX offline:            117.405 ms


In [42]:
print(f"ONNX speedup vs PyTorch:")
print(f"Online: {time_inference / time_onnx_online:.3f}x")
print(f"Offline {time_inference / time_onnx_offline:.3f}x")

ONNX speedup vs PyTorch:
Online: 1.446x
Offline 1.446x
